In [ ]:
# https://chatgpt.com/c/681a00ed-1bf4-8002-a650-068308b52566

import os
import json
import shutil
import pandas as pd
import numpy as np
import filecmp
from pathlib import Path

# Naudoja failą testavimui iš DUOMENYS_TST/records_npy_all/visi_zive_irasai.xlsx,
# visus įrašus iš aplanko DUOMENYS_TST/'records_zilvino_2025' ir apdoroja surastus
# atitinkamus įrašus iš DUOMENYS_TST/'records_npy_all' aplanko 

# Set display options
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', 1000)        # Set width to a high value to avoid line breaks
pd.set_option('display.max_colwidth', None) # Don't truncate column content

def zive_read_file_1ch(filename):
    """
    Reads a binary ECG file in Zive format and returns a zero-mean voltage signal in mV.
    """
    with open(filename, 'rb') as f:
        a = np.fromfile(f, dtype=np.dtype('>i4'))
    ADCmax = 0x800000
    Vref = 2.5
    b = (a - ADCmax / 2) * 2 * Vref / ADCmax / 3.5 * 1000
    return b - np.mean(b)

def get_the_row_by_filename(df, filename):
    """
    Extracts a dictionary of metadata from the DataFrame for a specific filename.
    """
    row = df[df['filename'] == filename].iloc[0]
    return row.to_dict()

def extract_user_rec(filename):
    """
    Splits filename in the form 'user_rec' to return user and recording numbers.
    """
    parts = filename.split('_')
    return int(parts[0]), int(parts[1])


# Duomenų aplankas su zive ir mit duomenimis
Duomenys = Path.home() / 'DI/ZIVEO_2025/DUOMENYS_UPD'

# Define paths
source_folder = Path(Duomenys, 'records_zilvino_2025')
target_folder = Path(Duomenys, 'records_npy_all')
excel_path = Path(Duomenys, 'records_npy_all/visi_zive_irasai.xlsx')

# Load or initialize Excel table
# df = pd.read_excel(excel_path)
df = pd.read_excel(excel_path, keep_default_na=False)
os.makedirs(target_folder, exist_ok=True)

# Gather existing entries
existing_filenames = df['filename'].tolist()
user_recordings = df.groupby('userId')['filename'].apply(list).to_dict()
all_user_numbers = [extract_user_rec(fn)[0] for fn in existing_filenames]
last_user_number = max(all_user_numbers, default=1000)

# Track processing
new_rows = []
list_of_all_filenames = []
list_of_new_filenames = []
changed_filenames = []  # Tracks new or updated filenames
processing_files = [f[:-5] for f in os.listdir(source_folder) if f.endswith('.json')]

print(f"Last user number in existing data: {last_user_number}")
print(f"\nProcessing files: {processing_files}")
print(f"Total files to process: {len(processing_files)}")

for base_name in processing_files:
    json_path = os.path.join(source_folder, base_name + '.json')
    data_path = os.path.join(source_folder, base_name)

    if not os.path.exists(data_path):
        continue

    with open(json_path, 'r') as f:
        meta = json.load(f)


    user_id, recording_id = meta['userId'], meta['recordingId']
    duplicate_rows = df[(df['userId'] == user_id) & (df['recordingId'] == recording_id)]

    if not duplicate_rows.empty:
        # ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++   This is an existing entry
        existing_filename = duplicate_rows.iloc[0]['filename']
        print(f"\nFound existing entry for recID {recording_id} with filename {existing_filename}")
        new_data_path = os.path.join(target_folder, existing_filename + '.npy')
        new_json_path = os.path.join(target_folder, existing_filename + '.json')

        # Skip if unchanged
        if os.path.exists(new_data_path) and os.path.exists(new_json_path) and \
           filecmp.cmp(json_path, new_json_path, shallow=False):
            print(f"Skipping unchanged record: {existing_filename}")
            list_of_all_filenames.append(existing_filename)
            continue

        # Backup old JSON if it's different
        if os.path.exists(new_json_path) and not filecmp.cmp(json_path, new_json_path, shallow=False):
            backup_folder = os.path.join(target_folder, 'backup_jsons1')
            os.makedirs(backup_folder, exist_ok=True)
            backup_path = os.path.join(backup_folder, existing_filename + '.json')
            shutil.copy(new_json_path, backup_path)
            print(f"Backed up old JSON to: {backup_path}")
        # Update JSON file
        shutil.copy(json_path, new_json_path)

        # Update DataFrame metadata
        noises = meta.get("noises_annotated", [])
        counts = meta.get("rpeakAnnotationCounts", {})
        index = duplicate_rows.index[0]

        df.at[index, 'noni'] = len(noises)
        df.at[index, 'N'] = counts.get("N", 0)
        df.at[index, 'S'] = counts.get("S", 0)
        df.at[index, 'V'] = counts.get("V", 0)
        df.at[index, 'U'] = counts.get("U", 0)

        list_of_all_filenames.append(existing_filename)
        changed_filenames.append(existing_filename)
        print(f"Updating json and metadata for {existing_filename}")
        existing_row = get_the_row_by_filename(df, existing_filename)
        print(f"{existing_row}")
        continue

    # ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++     New entry
    if user_id in user_recordings:
        user_files = user_recordings[user_id]
        user_numbers = [extract_user_rec(fn)[1] for fn in user_files]
        new_rec_num = max(user_numbers, default=-1) + 1
        new_user_num = extract_user_rec(user_files[0])[0]
    else:
        last_user_number += 1
        new_user_num = last_user_number
        new_rec_num = 0

    new_filename = f"{new_user_num}_{new_rec_num}"
    
    data = zive_read_file_1ch(data_path)
    np.save(os.path.join(target_folder, new_filename + '.npy'), data)
    shutil.copy(json_path, os.path.join(target_folder, new_filename + '.json'))

    length = len(data)
    noises = meta.get("noises_annotated", [])
    counts = meta.get("rpeakAnnotationCounts", {})

    new_row = {
        'filename': new_filename,
        'length': length,
        'quality': 0,
        'noni': int(len(noises)),
        'tag': 'NA',
        'mark': 'NA',
        'N': counts.get("N", 0),
        'S': counts.get("S", 0),
        'V': counts.get("V", 0),
        'U': counts.get("U", 0),
        'recordingId': recording_id,
        'userId': user_id,
        'basename': os.path.basename(data_path),
        'comment': meta.get('comment', '')
    }

    new_rows.append(new_row)
    changed_filenames.append(new_filename)
    list_of_all_filenames.append(new_filename)
    list_of_new_filenames.append(new_filename)

    # Update in-memory grouping
    user_recordings.setdefault(user_id, []).append(new_filename)
    print(f"\nProcessed: {data_path} as new record {new_filename}: {length} samples")

# Finalize DataFrame
if new_rows:
    df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
    # new_row = get_the_row_by_filename(df, existing_filename)
    # print(f"{new_row}")
    
# new_excel_path = excel_path.replace('.xlsx', '_new.xlsx')
# new_excel_path = str(excel_path).replace('.xlsx', '_new1.xlsx')
new_excel_path = excel_path
df.to_excel(new_excel_path, index=False)

# === Show changed rows before summary ===
print("\nChanged or newly added rows:")
# changed_rows = df[df['filename'].isin(changed_filenames)]
changed_rows = df[df['filename'].isin(changed_filenames)].sort_values(by='filename')
pd.set_option('display.max_columns', None)  # Optional: show all columns
print(changed_rows)

# Summary output
print(f"\nAll records: {len(list_of_all_filenames)}")
print(f"Processed {len(list_of_new_filenames)} new records.")
print("\nlist_of_all_filenames:", sorted(list_of_all_filenames))
print("list_of_new_filenames:", sorted(list_of_new_filenames))
print(f"New Excel file saved as: {new_excel_path}")

Last user number in existing data: 1103

Processing files: ['1626933.710', '1743959.255', '1626924.927', '1670188.752', '1742295.469', '1670177.650', '1743958.630', '1626931.201', '1670183.201']
Total files to process: 9

Found existing entry for recID 60f9188487cf6678b43780c3 with filename 1001_3
Skipping unchanged record: 1001_3

Found existing entry for recID 67f2cf89a588a8cdf08b3cfd with filename 1102_0
Skipping unchanged record: 1102_0

Found existing entry for recID 60f9188487cf669c7e3780b8 with filename 1001_2
Skipping unchanged record: 1001_2

Found existing entry for recID 638d8f609e4d4309eebbb0f1 with filename 1001_6
Skipping unchanged record: 1001_6

Found existing entry for recID 67d9543186cf219834695639 with filename 1103_0
Skipping unchanged record: 1103_0

Found existing entry for recID 638d8e519e4d43245abbb0c5 with filename 1001_7
Skipping unchanged record: 1001_7

Found existing entry for recID 67f2cf73f29283f5095d754d with filename 1102_1
Skipping unchanged record: 11